# XOR Neural Network in PyTorch

This notebook explains a small neural network that learns the **XOR** problem.

The network has:

- **2 input values**
- **2 hidden neurons**
- **1 output neuron**
- sigmoid activation functions
- mean squared error as the loss function
- stochastic gradient descent as the optimiser

The overall structure is:

(x_1, x_2) -> hidden neurons -> output neuron

## 1. Import the required PyTorch components

PyTorch is a machine-learning library.

We import:

- `torch` for tensors and numerical operations
- `torch.nn` for neural-network layers and loss functions
- `torch.optim` for optimisation algorithms that update the model parameters

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

## 2. Create the XOR dataset

XOR means **exclusive OR**.

Its rule is:

- output `0` when the two inputs are the same
- output `1` when the two inputs are different

The complete XOR truth table is:

| Input 1 | Input 2 | Target output |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

A PyTorch tensor is similar to a NumPy array. It stores numbers in a form that PyTorch can process efficiently.

In [3]:
# X contains the four XOR input patterns.
# Each row is one training example.
# Each training example contains two input values.
X = torch.tensor([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0]
])

# y contains the correct answer for each row in X.
# Each target is placed inside its own list so that y has shape (4, 1).
y = torch.tensor([
    [0.0],
    [1.0],
    [1.0],
    [0.0]
])

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: torch.Size([4, 2])
Shape of y: torch.Size([4, 1])


## 3. Define the neural-network class

A Python class is a blueprint for creating objects.

Here, `XORNet` describes the structure and behaviour of the neural network.

It inherits from `nn.Module`, which is the main PyTorch base class for neural-network models.

In [4]:
class XORNet(nn.Module):
    """A fully connected 2-input, 2-hidden, 1-output neural network."""

    def __init__(self):
        # Call the constructor of the parent class, nn.Module.
        # This allows PyTorch to track the layers and trainable parameters.
        super().__init__()

        # Create the hidden layer.
        #
        # nn.Linear(in_features=2, out_features=2) means:
        #   - each example supplies 2 input values;
        #   - the layer produces 2 output values;
        #   - therefore, the layer represents 2 hidden neurons.
        #
        # PyTorch automatically creates and stores:
        #   - 4 weights: 2 hidden neurons x 2 input values;
        #   - 2 biases: one bias for each hidden neuron.
        self.hidden = nn.Linear(2, 2)

        # Create the output layer.
        #
        # This layer receives the 2 hidden-layer outputs
        # and produces 1 final output value.
        #
        # PyTorch automatically creates and stores:
        #   - 2 weights: 1 output neuron x 2 hidden values;
        #   - 1 bias for the output neuron.
        self.output = nn.Linear(2, 1)

        # Create one sigmoid activation object.
        #
        # The name "sigmoid" is chosen by the programmer.
        # nn.Sigmoid is supplied by PyTorch.
        #
        # The same sigmoid object is used for both the hidden layer
        # and the output layer.
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        """Describe how input data moves through the network."""

        # First, self.hidden(x) performs the hidden layer's
        # weighted-sum calculation.
        #
        # If x has shape (4, 2), the result has shape (4, 2):
        # four examples and two hidden-neuron values per example.
        hidden_net_input = self.hidden(x)

        # Apply sigmoid separately to every hidden-neuron value.
        # The shape remains (4, 2).
        h = self.sigmoid(hidden_net_input)

        # Pass the two hidden outputs into the output layer.
        # The result has shape (4, 1).
        output_net_input = self.output(h)

        # Apply sigmoid to convert each output into a value
        # between 0 and 1.
        z = self.sigmoid(output_net_input)

        # Return the network's final predictions.
        return z

## 4. Create the model object

The statement below calls the class constructor, `__init__()`.

It creates the hidden layer, output layer, and sigmoid object.

It does **not** call `forward()` yet. The `forward()` method is called later when data is passed to the model using syntax such as `model(X)`.

In [4]:
# Fix the random seed so the initial parameter values are reproducible.
torch.manual_seed(0)

# Create one XORNet object.
# This calls XORNet.__init__().
model = XORNet()

print(model)

XORNet(
  (hidden): Linear(in_features=2, out_features=2, bias=True)
  (output): Linear(in_features=2, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


## 5. Inspect the model parameters before training

PyTorch creates the weights and biases automatically when each `nn.Linear` layer is constructed.

For a linear layer, PyTorch stores the weight matrix using this shape:

\[
(	ext{number of output units},\ 	ext{number of input values})
\]

Therefore:

- hidden-layer weights have shape `(2, 2)`
- hidden-layer biases have shape `(2,)`
- output-layer weights have shape `(1, 2)`
- output-layer bias has shape `(1,)`

In [5]:
print("Hidden-layer weight shape:", model.hidden.weight.shape)
print("Hidden-layer bias shape:", model.hidden.bias.shape)

print("Output-layer weight shape:", model.output.weight.shape)
print("Output-layer bias shape:", model.output.bias.shape)

Hidden-layer weight shape: torch.Size([2, 2])
Hidden-layer bias shape: torch.Size([2])
Output-layer weight shape: torch.Size([1, 2])
Output-layer bias shape: torch.Size([1])


## 6. Choose the loss function and optimiser

### Loss function

`nn.MSELoss()` calculates the mean squared difference between:

- the network's predictions
- the correct target values

A lower loss means the predictions are closer to the targets.

### Optimiser

`optim.SGD` means stochastic gradient descent.

The optimiser uses the gradients calculated during backpropagation to update all trainable weights and biases.

`lr=0.5` is the learning rate. It controls the size of each parameter update.

In [6]:
# Create the mean squared error loss function.
criterion = nn.MSELoss()

# model.parameters() gives the optimiser access to all
# trainable weights and biases in the network.
optimizer = optim.SGD(model.parameters(), lr=0.5)

## 7. Train the neural network

One complete pass through the training loop is called an **epoch**.

Each epoch performs five main steps:

1. clear old gradients
2. perform a forward pass
3. calculate the loss
4. perform backpropagation
5. update the weights and biases

In [7]:
number_of_epochs = 10000

for epoch in range(number_of_epochs):
    # Step 1: Clear gradients left over from the previous epoch.
    # PyTorch accumulates gradients by default, so they must
    # be cleared before calculating new ones.
    optimizer.zero_grad()

    # Step 2: Perform the forward pass.
    # Writing model(X) causes PyTorch to call model.forward(X).
    output = model(X)

    # Step 3: Compare the predictions with the correct targets.
    loss = criterion(output, y)

    # Step 4: Perform backpropagation.
    # PyTorch calculates the gradient of the loss with respect
    # to every trainable weight and bias.
    loss.backward()

    # Step 5: Update the weights and biases using SGD.
    optimizer.step()

    # Print progress once every 1000 epochs.
    if epoch % 1000 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}")

Epoch 0, Loss: 0.252217
Epoch 1000, Loss: 0.250001


Epoch 2000, Loss: 0.249970
Epoch 3000, Loss: 0.249944


Epoch 4000, Loss: 0.249887
Epoch 5000, Loss: 0.249702


Epoch 6000, Loss: 0.248540
Epoch 7000, Loss: 0.228302


Epoch 8000, Loss: 0.089208
Epoch 9000, Loss: 0.013919


## 8. Test the trained model

During testing, the network only makes predictions. It does not need to calculate gradients.

`torch.no_grad()` temporarily turns gradient tracking off.

The raw sigmoid outputs are numbers between 0 and 1.

They are converted into binary classes using a threshold:

- prediction greater than `0.5` becomes `1`
- otherwise it becomes `0`

In [8]:
with torch.no_grad():
    # Run the four XOR inputs through the trained model.
    predictions = model(X)

    # Convert probabilities into binary class predictions.
    binary_predictions = (predictions > 0.5).float()

    print("\nFinal raw outputs:")
    print(predictions)

    print("\nFinal binary predictions:")
    print(binary_predictions)

    print("\nCorrect targets:")
    print(y)


Final raw outputs:
tensor([[0.0858],
        [0.9254],
        [0.9254],
        [0.0875]])

Final binary predictions:
tensor([[0.],
        [1.],
        [1.],
        [0.]])

Correct targets:
tensor([[0.],
        [1.],
        [1.],
        [0.]])


## 9. Print the learned weights and biases

The values below were learned by the model during training.

In [9]:
print("Hidden-layer weights:")
print(model.hidden.weight.data)

print("\nHidden-layer biases:")
print(model.hidden.bias.data)

print("\nOutput-layer weights:")
print(model.output.weight.data)

print("\nOutput-layer bias:")
print(model.output.bias.data)

Hidden-layer weights:
tensor([[ 5.2577,  5.2641],
        [-5.1346, -5.1394]])

Hidden-layer biases:
tensor([-2.2488,  7.7258])

Output-layer weights:
tensor([[6.1807, 6.0051]])

Output-layer bias:
tensor([-8.9590])


## Summary

The program performs the following sequence:

1. creates the XOR input and target tensors;
2. defines a neural network with a `2 → 2 → 1` structure;
3. creates the model;
4. selects a loss function and optimiser;
5. repeatedly runs forward propagation and backpropagation;
6. updates the weights and biases;
7. tests whether the trained model has learned XOR.

The important distinction is:

```python
model = XORNet()
```

calls `__init__()` and creates the model, whereas:

```python
predictions = model(X)
```

causes PyTorch to call `forward(X)`.